<a href="https://colab.research.google.com/github/d03antony-ai/ma-strategie-vs-buyhold/blob/main/reseau_neurons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# ---------- Couche dense ----------
class Dense:
    def __init__(self, n_in, n_out):
        # Initialisation He (adaptée à ReLU)
        self.W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        self.b = np.zeros((1, n_out))

    def forward(self, x):
        self.x = x                       # cache pour le backward
        return x @ self.W + self.b

    def backward(self, grad_out, lr):
        # Gradients
        dW = self.x.T @ grad_out
        db = grad_out.sum(axis=0, keepdims=True)
        grad_in = grad_out @ self.W.T    # à propager à la couche précédente
        # Descente de gradient
        self.W -= lr * dW
        self.b -= lr * db
        return grad_in

# ---------- Activation ReLU ----------
class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, grad_out, lr):
        return grad_out * self.mask

# ---------- Perte MSE ----------
class MSE:
    def forward(self, pred, y):
        self.pred, self.y = pred, y
        return np.mean((pred - y) ** 2)

    def backward(self):
        return 2 * (self.pred - self.y) / self.y.shape[0]

# ---------- Réseau (backbone) ----------
class Network:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad, lr):
        for layer in reversed(self.layers):
            grad = layer.backward(grad, lr)

# ---------- Test : apprendre y = sin(x) ----------
if __name__ == "__main__":
    np.random.seed(0)
    X = np.linspace(-3, 3, 200).reshape(-1, 1)
    y = np.sin(X)

    net = Network([
        Dense(1, 32), ReLU(),
        Dense(32, 32), ReLU(),
        Dense(32, 1),
    ])
    loss_fn = MSE()

    for epoch in range(2000):
        pred = net.forward(X)
        loss = loss_fn.forward(pred, y)
        net.backward(loss_fn.backward(), lr=0.01)
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | loss = {loss:.5f}")




epoch    0 | loss = 13.45692
epoch  200 | loss = 0.14583
epoch  400 | loss = 0.11885
epoch  600 | loss = 0.08935
epoch  800 | loss = 0.06460
epoch 1000 | loss = 0.04757
epoch 1200 | loss = 0.03644
epoch 1400 | loss = 0.02832
epoch 1600 | loss = 0.02145
epoch 1800 | loss = 0.01656
